In [6]:
!pip install -q transformers datasets scikit-learn pandas torch


In [7]:
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import f1_score


In [13]:
from datasets import load_dataset, concatenate_datasets
import numpy as np
import pandas as pd

ds = load_dataset("unswnlporg/BESSTIE")
print(ds)   # shows splits

# Inspect each split separately
for split_name, split in ds.items():
    labels = np.array(split['label'])
    unique, counts = np.unique(labels, return_counts=True)
    print(f"Split: {split_name} — rows: {len(split)}")
    print("  Unique label values:", unique)
    print("  Counts per label:", dict(zip(unique, counts)))
    print()


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/17760 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2428 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'variety', 'source', 'task'],
        num_rows: 17760
    })
    validation: Dataset({
        features: ['text', 'label', 'variety', 'source', 'task'],
        num_rows: 2428
    })
})
Split: train — rows: 17760
  Unique label values: [0 1]
  Counts per label: {np.int64(0): np.int64(12092), np.int64(1): np.int64(5668)}

Split: validation — rows: 2428
  Unique label values: [0 1]
  Counts per label: {np.int64(0): np.int64(1653), np.int64(1): np.int64(775)}



In [14]:
label_map = {
    0: "negative",
    1: "positive"
}


In [15]:
candidate_labels = ["negative", "positive"]


In [16]:
hypothesis_template = "This text is {}."


In [ ]:
from transformers import pipeline
from sklearn.metrics import f1_score
import torch
import pandas as pd

device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

candidate_labels = ["negative", "positive"]

# Convert the validation split to a pandas DataFrame
df = ds["validation"].to_pandas()
# Map the numeric labels to text labels
df["label_text"] = df["label"].map(label_map)

results = []

for variety in ["en-AU", "en-IN", "en-UK"]:
    subset = df[df["variety"] == variety]

    texts = subset["text"].tolist()
    gold = subset["label_text"].tolist()

    preds = []
    for text in texts:
        out = classifier(
            text,
            candidate_labels,
            hypothesis_template="This text is {}."
        )
        preds.append(out["labels"][0])

    f1 = f1_score(gold, preds, average="macro")

    results.append({
        "model": "bart-large-mnli",
        "type": "prompt-based",
        "variety": variety,
        "macro_f1": f1
    })

    print(f"{variety}: macro-F1 = {f1:.4f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


en-AU: macro-F1 = 0.6094
en-IN: macro-F1 = 0.6260
en-UK: macro-F1 = 0.6610


In [ ]:
from transformers import pipeline
from sklearn.metrics import f1_score
import torch
import pandas as pd

device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

candidate_labels = ["negative", "positive"]

label_map = {
    0: "negative",
    1: "positive"
}

df = ds["validation"].to_pandas()
df["label_text"] = df["label"].map(label_map)

results = []

for variety in ["en-AU", "en-IN", "en-UK"]:
    subset = df[df["variety"] == variety]

    texts = subset["text"].tolist()
    gold = subset["label_text"].tolist()

    preds = []
    for text in texts:
        out = classifier(
            text,
            candidate_labels,
            hypothesis_template="This text is {}."
        )
        preds.append(out["labels"][0])

    f1 = f1_score(gold, preds, average="macro")

    results.append({
        "model": "bart-large-mnli",
        "type": "prompt-based",
        "variety": variety,
        "macro_f1": f1
    })

    print(f"{variety}: macro-F1 = {f1:.4f}")


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

en-AU: macro-F1 = 0.6094
en-IN: macro-F1 = 0.6260
en-UK: macro-F1 = 0.6610


In [ ]:
from transformers import pipeline
from sklearn.metrics import f1_score
import torch
import pandas as pd

device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "zero-shot-classification",
    model="roberta-large-mnli",
    device=device
)

candidate_labels = ["negative", "positive"]

label_map = {
    0: "negative",
    1: "positive"
}

df = ds["validation"].to_pandas()
df["label_text"] = df["label"].map(label_map)

results = []

for variety in ["en-AU", "en-IN", "en-UK"]:
    subset = df[df["variety"] == variety]

    texts = subset["text"].tolist()
    gold = subset["label_text"].tolist()

    preds = []
    for text in texts:
        out = classifier(
            text,
            candidate_labels,
            hypothesis_template="This text is {}."
        )
        preds.append(out["labels"][0])

    f1 = f1_score(gold, preds, average="macro")

    results.append({
        "model": "roberta-large-mnli",
        "type": "prompt-based",
        "variety": variety,
        "macro_f1": f1
    })

    print(f"{variety}: macro-F1 = {f1:.4f}")


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

en-AU: macro-F1 = 0.6130
en-IN: macro-F1 = 0.6143
en-UK: macro-F1 = 0.6550


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import f1_score
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

label_map = {
    0: "negative",
    1: "positive"
}

df = ds["validation"].to_pandas()
df["label_text"] = df["label"].map(label_map)

def predict_t5(text):
    prompt = (
        "Classify the sentiment of the following text as positive or negative.\n\n"
        f"Text: {text}\n\nLabel:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=3
    )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True).lower()

    if "positive" in pred:
        return "positive"
    elif "negative" in pred:
        return "negative"
    else:
        return "negative"  # safe fallback


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
results_t5 = []

for variety in ["en-AU", "en-IN", "en-UK"]:
    subset = df[df["variety"] == variety]

    texts = subset["text"].tolist()
    gold = subset["label_text"].tolist()

    preds = [predict_t5(text) for text in texts]

    f1 = f1_score(gold, preds, average="macro")

    results_t5.append({
        "model": "flan-t5-large",
        "type": "prompt-based",
        "variety": variety,
        "macro_f1": f1
    })

    print(f"{variety}: macro-F1 = {f1:.4f}")


en-AU: macro-F1 = 0.5809
en-IN: macro-F1 = 0.6251
en-UK: macro-F1 = 0.6548


In [ ]:
from transformers import pipeline
from sklearn.metrics import f1_score
import torch
import pandas as pd

device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
    device=device
)

candidate_labels = ["negative", "positive"]

label_map = {
    0: "negative",
    1: "positive"
}

df = ds["validation"].to_pandas()
df["label_text"] = df["label"].map(label_map)

results = []

for variety in ["en-AU", "en-IN", "en-UK"]:
    subset = df[df["variety"] == variety]

    texts = subset["text"].tolist()
    gold = subset["label_text"].tolist()

    preds = []
    for text in texts:
        out = classifier(
            text,
            candidate_labels,
            hypothesis_template="This text is {}."
        )
        preds.append(out["labels"][0])

    f1 = f1_score(gold, preds, average="macro")

    results.append({
        "model": "deberta-v3-large-mnli",
        "type": "prompt-based",
        "variety": variety,
        "macro_f1": f1
    })

    print(f"{variety}: macro-F1 = {f1:.4f}")


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight       

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


en-AU: macro-F1 = 0.5193
en-IN: macro-F1 = 0.4415
en-UK: macro-F1 = 0.4721


In [17]:
test_texts
test_labels
test_varieties


NameError: name 'test_texts' is not defined

In [ ]:
import pandas as pd


In [1]:
fine_tuned_results = [
    {"model": "bert-base-uncased", "paradigm": "Fine-tuned", "variety": "en-AU", "macro_f1": 0.551633},
    {"model": "bert-base-uncased", "paradigm": "Fine-tuned", "variety": "en-IN", "macro_f1": 0.559646},
    {"model": "bert-base-uncased", "paradigm": "Fine-tuned", "variety": "en-UK", "macro_f1": 0.630159},

    {"model": "roberta-base", "paradigm": "Fine-tuned", "variety": "en-AU", "macro_f1": 0.580728},
    {"model": "roberta-base", "paradigm": "Fine-tuned", "variety": "en-IN", "macro_f1": 0.619165},
    {"model": "roberta-base", "paradigm": "Fine-tuned", "variety": "en-UK", "macro_f1": 0.640697},

    {"model": "albert-base-v2", "paradigm": "Fine-tuned", "variety": "en-AU", "macro_f1": 0.569127},
    {"model": "albert-base-v2", "paradigm": "Fine-tuned", "variety": "en-IN", "macro_f1": 0.610124},
    {"model": "albert-base-v2", "paradigm": "Fine-tuned", "variety": "en-UK", "macro_f1": 0.646460},

    {"model": "deberta-v3-base", "paradigm": "Fine-tuned", "variety": "en-AU", "macro_f1": 0.382182},
    {"model": "deberta-v3-base", "paradigm": "Fine-tuned", "variety": "en-IN", "macro_f1": 0.417040},
    {"model": "deberta-v3-base", "paradigm": "Fine-tuned", "variety": "en-UK", "macro_f1": 0.411676},
]


In [2]:
prompt_based_results = [
    # NLI zero-shot
    {"model": "bart-large-mnli", "paradigm": "NLI zero-shot", "variety": "en-AU", "macro_f1": 0.6094},
    {"model": "bart-large-mnli", "paradigm": "NLI zero-shot", "variety": "en-IN", "macro_f1": 0.6260},
    {"model": "bart-large-mnli", "paradigm": "NLI zero-shot", "variety": "en-UK", "macro_f1": 0.6610},

    {"model": "roberta-large-mnli", "paradigm": "NLI zero-shot", "variety": "en-AU", "macro_f1": 0.6130},
    {"model": "roberta-large-mnli", "paradigm": "NLI zero-shot", "variety": "en-IN", "macro_f1": 0.6143},
    {"model": "roberta-large-mnli", "paradigm": "NLI zero-shot", "variety": "en-UK", "macro_f1": 0.6550},

    {"model": "deberta-v3-large (Laurer)", "paradigm": "NLI zero-shot", "variety": "en-AU", "macro_f1": 0.5809},
    {"model": "deberta-v3-large (Laurer)", "paradigm": "NLI zero-shot", "variety": "en-IN", "macro_f1": 0.6251},
    {"model": "deberta-v3-large (Laurer)", "paradigm": "NLI zero-shot", "variety": "en-UK", "macro_f1": 0.6548},

    # Non-NLI / instruction
    {"model": "deberta-v3-large (vanilla)", "paradigm": "Non-NLI pseudo zero-shot", "variety": "en-AU", "macro_f1": 0.5193},
    {"model": "deberta-v3-large (vanilla)", "paradigm": "Non-NLI pseudo zero-shot", "variety": "en-IN", "macro_f1": 0.4415},
    {"model": "deberta-v3-large (vanilla)", "paradigm": "Non-NLI pseudo zero-shot", "variety": "en-UK", "macro_f1": 0.4721},

    {"model": "flan-t5-large", "paradigm": "Instruction prompting", "variety": "en-AU", "macro_f1": 0.4681},
    {"model": "flan-t5-large", "paradigm": "Instruction prompting", "variety": "en-IN", "macro_f1": 0.4643},
    {"model": "flan-t5-large", "paradigm": "Instruction prompting", "variety": "en-UK", "macro_f1": 0.4120},
]


In [8]:
df = pd.DataFrame(fine_tuned_results + prompt_based_results)
df


,model,paradigm,variety,macro_f1
0,bert-base-uncased,Fine-tuned,en-AU,0.551633
1,bert-base-uncased,Fine-tuned,en-IN,0.559646
2,bert-base-uncased,Fine-tuned,en-UK,0.630159
3,roberta-base,Fine-tuned,en-AU,0.580728
4,roberta-base,Fine-tuned,en-IN,0.619165
5,roberta-base,Fine-tuned,en-UK,0.640697
6,albert-base-v2,Fine-tuned,en-AU,0.569127
7,albert-base-v2,Fine-tuned,en-IN,0.610124
8,albert-base-v2,Fine-tuned,en-UK,0.646460
9,deberta-v3-base,Fine-tuned,en-AU,0.382182


In [9]:
final_table = df.pivot_table(
    index=["model", "paradigm"],
    columns="variety",
    values="macro_f1"
).reset_index()

final_table


variety,model,paradigm,en-AU,en-IN,en-UK
0,albert-base-v2,Fine-tuned,0.569127,0.610124,0.646460
1,bart-large-mnli,NLI zero-shot,0.609400,0.626000,0.661000
2,bert-base-uncased,Fine-tuned,0.551633,0.559646,0.630159
3,deberta-v3-base,Fine-tuned,0.382182,0.417040,0.411676
4,deberta-v3-large (Laurer),NLI zero-shot,0.580900,0.625100,0.654800
5,deberta-v3-large (vanilla),Non-NLI pseudo zero-shot,0.519300,0.441500,0.472100
6,flan-t5-large,Instruction prompting,0.468100,0.464300,0.412000
7,roberta-base,Fine-tuned,0.580728,0.619165,0.640697
8,roberta-large-mnli,NLI zero-shot,0.613000,0.614300,0.655000


In [12]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [19]:
from datasets import load_dataset

dataset = load_dataset("unswnlporg/BESSTIE")

val_data = dataset["validation"]

texts = val_data["text"]
labels = val_data["label"]
varieties = val_data["variety"]


In [20]:
predictions = []

for text in texts:
    result = classifier(text, candidate_labels)
    top_label = result["labels"][0]
    predictions.append(1 if top_label == "positive" else 0)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
macro_f1 = f1_score(labels, predictions, average="macro")
print("XLM-R Zero-Shot Macro F1:", macro_f1)


In [21]:
import pandas as pd

results = {}

for variety in ["en-AU", "en-IN", "en-UK"]:
    indices = [i for i, v in enumerate(varieties) if v == variety]

    y_true = [labels[i] for i in indices]
    y_pred = [predictions[i] for i in indices]

    results[variety] = f1_score(y_true, y_pred, average="macro")

print("Per-variety F1:", results)


Per-variety F1: {'en-AU': 0.5736004422718808, 'en-IN': 0.5674167292626018, 'en-UK': 0.626281389748882}


In [24]:
final_table["mean_f1"] = final_table[["en-AU", "en-IN", "en-UK"]].mean(axis=1)

final_table


,model,paradigm,en-AU,en-IN,en-UK,mean_f1
0,albert-base-v2,Fine-tuned,0.569127,0.610124,0.646460,0.608570
1,bart-large-mnli,NLI zero-shot,0.609400,0.626000,0.661000,0.632133
2,bert-base-uncased,Fine-tuned,0.551633,0.559646,0.630159,0.580479
3,deberta-v3-base,Fine-tuned,0.382182,0.417040,0.411676,0.403633
4,deberta-v3-large (Laurer),NLI zero-shot,0.580900,0.625100,0.654800,0.620267
5,deberta-v3-large (vanilla),Non-NLI pseudo zero-shot,0.519300,0.441500,0.472100,0.477633
6,flan-t5-large,Instruction prompting,0.468100,0.464300,0.412000,0.448133
7,roberta-base,Fine-tuned,0.580728,0.619165,0.640697,0.613530
8,roberta-large-mnli,NLI zero-shot,0.613000,0.614300,0.655000,0.627433
9,xlm-roberta-large-xnli,NLI zero-shot,0.573600,0.567417,0.626281,0.589100


In [30]:
final_table[final_table["paradigm"] == "NLI zero-shot"]


,model,paradigm,en-AU,en-IN,en-UK,mean_f1
1,bart-large-mnli,NLI zero-shot,0.6094,0.626000,0.661000,0.632133
4,deberta-v3-large (Laurer),NLI zero-shot,0.5809,0.625100,0.654800,0.620267
8,roberta-large-mnli,NLI zero-shot,0.6130,0.614300,0.655000,0.627433
9,xlm-roberta-large-xnli,NLI zero-shot,0.5736,0.567417,0.626281,0.589100
10,xlm-roberta-large-xnli,NLI zero-shot,0.5736,0.567417,0.626281,0.589100
11,xlm-roberta-large-xnli,NLI zero-shot,0.5736,0.567417,0.626281,0.589100
12,xlm-roberta-large-xnli,NLI zero-shot,0.5736,0.567417,0.626281,0.589100


In [31]:
final_table = final_table.drop_duplicates(subset=["model"], keep="first")


In [32]:
final_table["mean_f1"] = final_table[["en-AU","en-IN","en-UK"]].mean(axis=1)


/tmp/ipython-input-2794484591.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table["mean_f1"] = final_table[["en-AU","en-IN","en-UK"]].mean(axis=1)


In [35]:
fine_scores = final_table[final_table["paradigm"] == "Fine-tuned"]["mean_f1"].values
nli_scores = final_table[final_table["paradigm"] == "NLI zero-shot"]["mean_f1"].values

print("Fine length:", len(fine_scores))
print("NLI length:", len(nli_scores))


Fine length: 4
NLI length: 4


In [36]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(fine_scores, nli_scores, equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)


t-statistic: -1.2935768388439925
p-value: 0.28063628891952247


In [33]:
from scipy.stats import ttest_ind

final_table["mean_f1"] = final_table[["en-AU","en-IN","en-UK"]].mean(axis=1)

fine_scores = final_table[final_table["paradigm"] == "Fine-tuned"]["mean_f1"].values
nli_scores = final_table[final_table["paradigm"] == "NLI zero-shot"]["mean_f1"].values

print("Fine-tuned:", fine_scores)
print("NLI zero-shot:", nli_scores)

t_stat, p_value = ttest_ind(fine_scores, nli_scores, equal_var=False)

print("\nWelch Independent t-test (4 vs 4)")
print("t-statistic:", t_stat)
print("p-value:", p_value)


Fine-tuned: [0.60857033 0.58047933 0.40363267 0.61353   ]
NLI zero-shot: [0.63213333 0.62026667 0.62743333 0.58909952]

Welch Independent t-test (4 vs 4)
t-statistic: -1.2935768388439925
p-value: 0.28063628891952247


/tmp/ipython-input-685742647.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table["mean_f1"] = final_table[["en-AU","en-IN","en-UK"]].mean(axis=1)


In [37]:
final_table


,model,paradigm,en-AU,en-IN,en-UK,mean_f1
0,albert-base-v2,Fine-tuned,0.569127,0.610124,0.646460,0.608570
1,bart-large-mnli,NLI zero-shot,0.609400,0.626000,0.661000,0.632133
2,bert-base-uncased,Fine-tuned,0.551633,0.559646,0.630159,0.580479
3,deberta-v3-base,Fine-tuned,0.382182,0.417040,0.411676,0.403633
4,deberta-v3-large (Laurer),NLI zero-shot,0.580900,0.625100,0.654800,0.620267
5,deberta-v3-large (vanilla),Non-NLI pseudo zero-shot,0.519300,0.441500,0.472100,0.477633
6,flan-t5-large,Instruction prompting,0.468100,0.464300,0.412000,0.448133
7,roberta-base,Fine-tuned,0.580728,0.619165,0.640697,0.613530
8,roberta-large-mnli,NLI zero-shot,0.613000,0.614300,0.655000,0.627433
9,xlm-roberta-large-xnli,NLI zero-shot,0.573600,0.567417,0.626281,0.589100
